# 03. Data Cleaning
##### DSR Berlin | Data Science Fundamentals
##### Inspired by Adam Green (adgefficiency.com), Rachel Berryman, and Samson Afolabi

---

## 🎯 Learning Objectives
By the end of this notebook, you will be able to:
1. Characterize datasets across 6 critical dimensions: **Quality, Quantity, Diversity, Cardinality, Dimensionality, and Sparsity**.
2. Enforce the **"Split First" golden rule** to isolate the test set and guarantee **zero data leakage**.
3. Evaluate **duplicates** with mathematical rigor: explain why duplicates between train and test are lethal, but why "true" duplicates inside training data provide vital empirical weight of evidence.
4. Detect disguised **missing values** (whitespace strings, sentinel `-999`, null tokens) and identify missingness mechanisms (**MCAR, MAR, MNAR**).
5. Apply four distinct imputation strategies: row/column drops, simple statistics, conditional group statistics, and model-based imputation (`KNNImputer`).
6. Detect and remediate **outliers** using Z-scores, normal distribution limits, and IQR percentile clipping.
7. Clean noisy string columns and leverage modern LLM-assisted cleaning patterns responsibly.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import zscore
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer, KNNImputer

import warnings

warnings.filterwarnings("ignore")

# Set plotting style and random seed
plt.style.use(
    "seaborn-v0_8-whitegrid"
    if "seaborn-v0_8-whitegrid" in plt.style.available
    else "default"
)
RANDOM_SEED = 42

## 1. Data Characterization: The Six Dimensions

Before writing a single line of cleaning code, a senior data scientist characterizes the raw dataset across six foundational dimensions:

<div align="center">
  <img src="../images/data_characterization.jpg" width="650" alt="Data Characterization" />
  <p><em>Figure 1: Six Key Dimensions of Data Characterization.</em></p>
</div>

1. **Quality**:
   - Are there recording errors, impossible negative numbers, disguised nulls, or typos?
2. **Quantity**:
   - How many rows ($N$) and columns ($P$) exist? Is the sample size sufficient relative to model parameters?
3. **Diversity & Distribution Shift**:
   - Does the distribution of training samples match the distribution of the test set and real-world deployment?
4. **Cardinality**:
   - The number of unique values in categorical features. Low cardinality (`Contract`: 3 values) vs high cardinality (`ZipCode`: 5,000 values).
5. **Dimensionality**:
   - The ratio of features to samples ($P / N$). When $P \approx N$ or $P > N$, models suffer from the **curse of dimensionality** and severe overfitting.
6. **Sparsity**:
   - The proportion of zeros or empty entries in your data matrix. Text TF-IDF matrices or one-hot encodings often exceed 99% sparsity.


## 2. Ingesting the Raw Project Dataset

Let's load `../data/telco_customer_churn_raw.csv`. This dataset represents customer records for a telecommunications provider where we want to predict whether a customer will cancel their subscription (`Churn`: Yes/No).


In [ ]:
raw_path = '../data/telco_customer_churn_raw.csv'

# TODO: Load raw dataset via pd.read_csv and display shape and first 3 rows
# YOUR CODE HERE


In [ ]:
# TODO: Check column data types and non-null counts using df_raw.info()
# YOUR CODE HERE


## 3. The Golden Rule: Split First (Zero Data Leakage)

> 🛑 **NEVER clean, impute, scale, or transform your entire dataset before splitting!**

If you compute the mean of `MonthlyCharges` on the whole dataset to fill missing values, **the test set's values have contaminated your training statistics**. This is known as **Data Leakage**. Your test evaluation will look artificially optimistic, but the model will degrade in real-world production.

<div align="center">
  <img src="../images/train_val_test_split.jpg" width="650" alt="Train Test Validation Split" />
  <p><em>Figure 2: Isolating the Test Set Immediately.</em></p>
</div>

### Immediate Isolation Protocol:
1. Split into **Train (80%)** and **Test (20%)** right now.
2. Put the Test set in a "vault".
3. Compute all imputation statistics, encodings, and scaling parameters **solely on Train**.
4. Apply the fitted transformations to Test without refitting.


In [ ]:
# TODO: Separate Features (X) and Target (y), then perform stratified train_test_split (80/20)
# YOUR CODE HERE


## 4. Handling Duplicates: Nuance, Evidence, and Leakage

Duplicates are not simply rows to delete mindlessly. We must distinguish between **two fundamentally different types of duplicates**:

### 1. Accidental / Cross-Split Duplicates (Lethal Leakage):
- Caused by logging retries, double ETL loads, or bad database merges.
- If an exact duplicate row appears in both **Train** and **Test**, your test metric is fraudulent: the model merely memorized a sample it saw during training!
- *Action*: Drop identical records.

### 2. "True" Feature Duplicates within Training Data (Statistical Evidence):
- Imagine two customers who both have: `gender='Female'`, `SeniorCitizen=0`, `Contract='Month-to-month'`, `InternetService='DSL'`.
- Customer A churns, Customer B does not churn.
- Or ten distinct customers have identical profiles and all ten churn!
- In Bayesian statistics and probabilistic ML, **frequency information is genuine signal**. These repetitions provide empirical *weight of evidence*. If you deduplicate purely based on feature profiles, you distort the underlying prior probability $P(X)$!


In [ ]:
# TODO: Check for exact full-row duplicates in Train using .duplicated() and remove from training partition
# YOUR CODE HERE


## 5. Missing Values: Detection, Mechanisms & Solutions

### Missingness is NOT always `np.nan`!
In real production systems, missing values frequently masquerade as:
- Empty string whitespace: `' '` or `''`
- Sentinel numbers: `-999`, `-1`, `0`
- String tokens: `'?'`, `'Unknown'`, `'N/A'`, `'null'`

### Why is Data Missing? The 3 Mechanisms:
1. **MCAR (Missing Completely at Random)**:
   - Missingness has zero correlation with any feature (e.g. random sensor glitch).
   - Safe to drop or simple-impute without introducing statistical bias.
2. **MAR (Missing at Random)**:
   - Missingness depends on *another observed feature*.
   - *Example*: Younger participants skip an income question on a digital survey at higher rates, but within the younger age group itself, the missingness of income is random.
3. **MNAR (Missing Not at Random)**:
   - Missingness depends on the unobserved/missing value itself (e.g. high-income earners refusing to disclose their wealth).
   - Requires adding a missing indicator flag (`MissingIndicator`).


In [ ]:
# TODO: Detect disguised missing values in X_train:
# 1. Standard .isna() check
# 2. Whitespace ' ' in 'TotalCharges'
# 3. Sentinel -999.0 in 'MonthlyCharges'
# 4. '?' or 'Unknown' in 'PaymentMethod'
# YOUR CODE HERE


In [ ]:
def standardize_missing(df):
    # TODO: Replace ' ' in TotalCharges with np.nan and cast to numeric
    # TODO: Replace negative sentinels in MonthlyCharges with np.nan
    # TODO: Replace '?' and 'Unknown' in PaymentMethod with np.nan
    # YOUR CODE HERE
    pass

# X_train = standardize_missing(X_train)
# X_test = standardize_missing(X_test)


### Imputation Strategies:

#### Strategy 1: Simple Statistical Imputation (Median / Mode)
- **Continuous**: Fill with median (robust to skew and outliers).
- **Categorical**: Fill with mode (most frequent category).
- *Rule*: Fit the imputer on `X_train`, transform `X_train` and `X_test`!


In [ ]:
# TODO: Compute training statistics (median MonthlyCharges, mode PaymentMethod)
# YOUR CODE HERE


#### Strategy 2: Conditional Group Imputation
Instead of a blunt global median, fill missing values conditional on another correlated feature.
For example, customers with a **Fiber optic** internet service pay significantly higher monthly charges than **DSL** customers. Imputing based on `InternetService` retains feature variance!


In [ ]:
# TODO: Compute conditional median of MonthlyCharges grouped by InternetService
# TODO: Impute MAR TotalCharges: when tenure == 0 -> TotalCharges = 0.0
# TODO: Impute missing MonthlyCharges conditionally (fallback to global median)
# TODO: Fill missing PaymentMethod with training mode
# YOUR CODE HERE


## 6. Outlier Detection: Z-Scores, IQR & Clipping

Outliers arise from two distinct sources:
1. **Measurement / Recording Errors** (e.g. `tenure = 1200` months when a human has only been alive 100 years, or `MonthlyCharges = 999.0` fat-finger entry). $\rightarrow$ **Must be corrected or dropped**.
2. **True Extreme Power Users** (e.g. a corporate enterprise account spending 5x the average). $\rightarrow$ **Preserved, but clipped or log-transformed**.

<div align="center">
  <img src="../images/zscore_outliers_normal.png" width="650" alt="Z-Score Outliers" />
  <p><em>Figure 3: Normal distribution curve showing z-score outlier boundaries ($|z| > 3$).</em></p>
</div>

### Detection Method 1: Z-Score ($|z| > 3.0$)
The Z-score measures how many standard deviations a point lies from the mean:
$$z = \frac{x - \mu}{\sigma}$$
Under a normal distribution, $99.73\%$ of observations fall within $|z| \le 3$. Any point with $|z| > 3$ is a candidate outlier.


In [ ]:
# TODO: Detect outliers on MonthlyCharges and tenure using Z-score (|z| > 3.0)
# YOUR CODE HERE


### Detection Method 2: Tukey's Interquartile Range (IQR) & Winsorization (Clipping)

For skewed non-normal distributions, the IQR is vastly superior to the Z-score because the median and quartiles are robust to the outliers themselves:

$$\text{IQR} = Q_3 - Q_1$$
$$\text{Upper Fence} = Q_3 + 1.5 \times \text{IQR}$$
$$\text{Lower Fence} = Q_1 - 1.5 \times \text{IQR}$$

### Winsorization (Clipping):
Instead of discarding rows (which permanently discards valuable sample size), we **cap / clip** extreme values at a sensible threshold.


In [ ]:
# TODO: Apply domain clipping using .clip() (tenure [0, 72], MonthlyCharges [15, 150])
# YOUR CODE HERE


## 7. String Cleaning & Categorical Standardization

Text data in categorical columns often suffers from leading/trailing whitespace, mixed casing, or multiple spellings of the same underlying category.


In [ ]:
# TODO: Clean string columns (strip whitespace and title case)
# YOUR CODE HERE


## 8. Persisting Cleaned Partitions

Now that our dataset has been rigorously cleaned with zero data leakage, we persist `X_train_clean.csv`, `X_test_clean.csv`, `y_train.csv`, and `y_test.csv` in `../data/`.
These cleaned partitions will be directly picked up by **Notebook 04: Feature Engineering**!


In [ ]:
# TODO: Save cleaned training and test partitions to ../data/
# X_train.to_csv('../data/telco_X_train_clean.csv', index=False)
# X_test.to_csv('../data/telco_X_test_clean.csv', index=False)
# y_train.to_csv('../data/telco_y_train.csv', index=False)
# y_test.to_csv('../data/telco_y_test.csv', index=False)
# YOUR CODE HERE
